# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR\u00b2 dataset using the `mlcroissant` library, which leverages the Croissant standard for describing ML-ready datasets.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
# For basic visualization use
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show metadata summary
md = dataset.metadata
print(f"Name: {md.name}")
print(f"Identifier: {md.identifier}")
print(f"Description: {md.description}")
print(f"Published: {getattr(md, 'datePublished', 'N/A')}")
print(f"Authors: {getattr(md, 'author', 'N/A')}")
print(f"Keywords: {getattr(md, 'keywords', 'N/A')}")

## 2. Data Overview
Review the available record sets and their fields. Entities are referenced by `@id` according to the Croissant specification.

In [ ]:
# List all available record sets in the dataset by their @id
print('Available record sets:')
for record_set in dataset.metadata.record_sets:
    print(f" - Record Set @id: {record_set['@id']}")

# For this dataset, let's inspect the first record set (main data table):
main_record_set = dataset.metadata.record_sets[0]['@id']
print(f"Main data is in Record Set: {main_record_set}")

# List fields and columns for this record set (by @id and name)
main_rs = next(rs for rs in dataset.metadata.record_sets if rs['@id'] == main_record_set)
if 'fields' in main_rs:
    print('Fields in main record set:')
    for field in main_rs['fields']:
        print(f" - Field @id: {field['@id']}  |  Name: {field['name']}")
else:
    print('No fields listed for main record set.')

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All references to record sets and fields use their full `@id`.

In [ ]:
# Collect all record sets by @id for flexible loading
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show the columns in the main record set
main_df = dataframes[main_record_set]
print(f"Columns in main record set ({main_record_set}):")
print(main_df.columns.tolist())

# Preview the first few rows
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps using field `@id`s. This includes filtering by a quantitative field, normalization, and grouping operations. See Croissant field/column metadata for valid `@id`s.

In [ ]:
# Choose numeric and categorical fields for analysis, by `@id`
# Replace these with the proper field @id as per the dataset schema (see Section 2 output)
numeric_field_id = None
group_field_id = None

# Automatically try to pick a numeric field (e.g., MW, molecular weight, peptide_count, etc.)
import numpy as np
for col in main_df.columns:
    # Attempt to guess if it's numeric by sampling the column
    # Croissant will often use field @id as the column name
    vals = main_df[col].dropna()
    if len(vals) > 0:
        try:
            float(vals.iloc[0])
            numeric_field_id = col
            break
        except Exception:
            continue

# Try to pick a group field (e.g., group/samples/condition/experiment field @id)
for col in main_df.columns:
    if col != numeric_field_id and main_df[col].dtype == object and main_df[col].nunique() < 20:
        group_field_id = col
        break

print(f"Using numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}")

# Filter based on the numeric field (> threshold)
threshold = None
if numeric_field_id is not None:
    # Try to convert the column to numeric (handles str columns)
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    # Set threshold as median
    threshold = main_df[numeric_field_id].median()
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold} (median): {len(filtered_df)} records")

    # Normalize column
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (if available)
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
else:
    print('No numeric field found automatically. Please review earlier output and set `numeric_field_id` to the desired column.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plot histogram of the numeric field (if extracted above)
if numeric_field_id is not None and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    main_df[numeric_field_id].hist(bins=40)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # Boxplot by group field if available
    if group_field_id is not None:
        plt.figure(figsize=(10, 6))
        main_df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.suptitle("")
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('No numeric field found for plotting.')

## 6. Conclusion
In this notebook, we loaded, overviewed, and performed basic exploratory analysis on the FAIR\u00b2 mass spectrometry protein dataset using the Croissant schema with the `mlcroissant` library. The use of explicit `@id` fields for record sets and columns ensures reproducibility and transparency. Further data analysis or modeling workflows can be built on top of this foundation.